# TxGNN — Google Colab setup

This notebook installs the **exact 2020-era stack** TxGNN needs, on Colab.

Why it is not one `pip install`: today's Colab ships **Python 3.11/3.12**, but
**DGL 0.5.2** and **PyTorch 1.8** only have wheels for **Python ≤ 3.8**. So we first
put a Python 3.8 environment on Colab with `condacolab`, then install the matched
CUDA 10.2 stack (works on Colab's free **T4** GPU).

| Component | Version | Source |
|---|---|---|
| Python | 3.8 | condacolab (Miniconda 4.12.0) |
| PyTorch | 1.8.1 + cu102 | download.pytorch.org |
| DGL | 0.5.2 (cu102) | PyPI `dgl-cu102` |
| PyG | scatter 2.0.8 / sparse 0.6.12 / geometric 2.0.4 | data.pyg.org (only needed for disease-area splits) |
| pandas / numpy / scipy / sklearn / matplotlib / tqdm / goatools | pinned | requirements.txt |

**Run the cells top to bottom.** Cell 2 restarts the kernel — that is expected; just
continue from cell 3 afterwards. Set Runtime → Change runtime type → **GPU** first.

### 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi

### 2. Install a Python 3.8 base (condacolab)
This **restarts the kernel automatically**. That is normal — do not re-run this cell;
just move to cell 3 after the restart message.

We use `install_from_url(...)` with the Python 3.8 Miniconda installer instead of
`condacolab.install_miniconda()`, because the latter currently crashes with
`💥💔💥 Checksum failed!` — condacolab pins a stale installer hash (a known,
widespread bug). `install_from_url` does **no** checksum check, so it sidesteps the
bug while doing the same kernel-restart / PATH setup.

In [ ]:
!pip install -q condacolab
import condacolab
# install_from_url does NO checksum verification, so it avoids the stale-hash crash
# in condacolab.install_miniconda(). The py38 installer gives the Python 3.8 base we need.
condacolab.install_from_url(
    "https://repo.anaconda.com/miniconda/Miniconda3-py38_4.12.0-Linux-x86_64.sh"
)

### 3. Verify the environment (run after the restart)
You should see `Python 3.8.x`. If it is not 3.8, re-run cell 2 once.

In [ ]:
import condacolab
condacolab.check()
!python --version

### 4. Get the TxGNN code
Change `REPO_URL` to your own fork if you have one (so it uses the pinned
`requirements.txt`). The upstream repo works too — we pin every dependency
explicitly below regardless.

In [ ]:
REPO_URL = "https://github.com/mims-harvard/TxGNN.git"  # <-- change to your fork if needed
!git clone $REPO_URL
%cd TxGNN

### 5. Install PyTorch + DGL + PyG (CUDA 10.2)
These use custom wheel indexes and must be installed before the plain deps.

In [ ]:
# PyTorch 1.8.1 (CUDA 10.2) — in the ~1.8–1.12 range, matched to DGL 0.5.2
!pip install torch==1.8.1+cu102 -f https://download.pytorch.org/whl/torch_stable.html

# DGL 0.5.2 (CUDA 10.2)
!pip install dgl-cu102==0.5.2

# PyG — only needed for disease-area splits; wheels matched to torch 1.8.0+cu102
!pip install torch-scatter==2.0.8 torch-sparse==0.6.12 -f https://data.pyg.org/whl/torch-1.8.0+cu102.html
!pip install torch-geometric==2.0.4

### 6. Install the pinned Python deps + TxGNN
`--no-deps` on TxGNN stops pip from “upgrading” pandas/numpy back to versions that
break the code (the repo uses `DataFrame.append`, removed in pandas 2.0).

In [ ]:
!pip install numpy==1.21.6 pandas==1.3.5 scipy==1.7.3 scikit-learn==1.0.2 \
             matplotlib==3.5.3 tqdm==4.64.1 goatools==1.2.3 requests==2.28.2
!pip install -e . --no-deps

### 7. Sanity check

In [ ]:
import torch, dgl, numpy, pandas, sklearn, scipy, matplotlib, tqdm
from txgnn import TxData, TxGNN, TxEval
print("python  ", __import__("sys").version.split()[0])
print("torch   ", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("dgl     ", dgl.__version__)
print("numpy   ", numpy.__version__)
print("pandas  ", pandas.__version__)
print("sklearn ", sklearn.__version__)
print("TxGNN imports OK")

### 8. Minimal end-to-end smoke test
Downloads the knowledge graph, builds the `complex_disease` split, and runs a tiny
pretrain to confirm the GPU path works. (First run downloads the data, ~a few minutes.)

In [ ]:
TxData = TxData(data_folder_path = './data')
TxData.prepare_split(split = 'complex_disease', seed = 42)

TxGNN = TxGNN(data = TxData,
              weight_bias_track = False,
              proj_name = 'TxGNN',
              exp_name = 'TxGNN',
              device = 'cuda:0')

TxGNN.model_initialize(n_hid = 100, n_inp = 100, n_out = 100,
                       proto = True, proto_num = 3, attention = False,
                       sim_measure = 'all_nodes_profile',
                       agg_measure = 'rarity', num_walks = 200, path_length = 2)

# tiny run just to confirm the pipeline executes end-to-end
TxGNN.pretrain(n_epoch = 1, learning_rate = 1e-3, batch_size = 1024, train_print_per_n = 20)
print('Smoke test finished.')

---
## Optional: fast smoke test on a mini KG

The full KG is **8.1M edges / ~945 MB**, so one end-to-end run takes hours. To check the
code works before committing to a real run, `make_mini_kg.py` carves out a small
connectivity-aware subgraph (~315k edges) that keeps the same schema and relation
vocabulary, so it exercises the same code paths in about a minute.

The subset is built to stay valid for the pipeline: every edge is kept in **both
directions** (`preprocess_kg` de-duplicates by orientation), drug-disease edges are never
downsampled (they are the task), and enough treated diseases are kept for the 5% test
split to be non-empty.

In [ ]:
# download the full KG once (~945 MB), then carve out the mini subset
import os
os.makedirs('data_full', exist_ok=True)
!wget -q --show-progress -O data_full/kg.csv    https://dataverse.harvard.edu/api/access/datafile/7144484
!wget -q --show-progress -O data_full/node.csv  https://dataverse.harvard.edu/api/access/datafile/7144482
!wget -q --show-progress -O data_full/edges.csv https://dataverse.harvard.edu/api/access/datafile/7144483

!python make_mini_kg.py --src data_full/kg.csv --out data_mini --n-diseases 400 --seed 42

In [ ]:
# whole pipeline on the mini KG: split -> DGL graph -> pretrain -> finetune -> eval
!python test_mini_pipeline.py

Expect near-chance metrics here — 1 pretrain epoch + 2 finetune epochs on a small
subgraph is far too little to learn anything real. This test answers *"does the code
run correctly end to end?"*, not *"is the model good?"*. For real numbers, train on the
full KG with the paper's epoch counts:

```python
TxDataObj = TxData(data_folder_path='./data')   # downloads the full KG
TxDataObj.prepare_split(split='complex_disease', seed=42)
# ... then TxGNN.pretrain(n_epoch=2) and TxGNN.finetune(n_epoch=500)
```